# Train FastViT Detector trên BDD100K

Notebook này thực hiện trọn pipeline: kiểm tra môi trường, tải BDD100K, giải nén, chuyển nhãn detection sang COCO, kiểm tra/hiển thị dữ liệu, tải và kiểm tra pretrained backbone, train FastViT và đánh giá checkpoint tốt nhất. Cấu hình full run bám theo `docs/usage_guide.md` (`fixed_c5 + qfl`).

- Toàn bộ dữ liệu, log, checkpoint và kết quả được lưu dưới `/mnt/bdd100k`.
- Notebook dùng mã nguồn trong repo `ml-fastvit`; hãy chạy kernel từ repo hoặc chỉnh `REPO_ROOT_OVERRIDE` ở cell cấu hình.
- BDD100K khá lớn. Nên dùng máy Linux có GPU NVIDIA và còn tối thiểu khoảng 30 GB dung lượng trống.

## 1. Cấu hình

Các giá trị mặc định bên dưới là cấu hình khuyến nghị đầy đủ trong `docs/usage_guide.md`. Nếu thiếu VRAM, giảm `BATCH_SIZE` xuống 4/2 và tăng `ACCUM_STEPS` để giữ effective batch size. `EXISTING_ARCHIVE` có thể trỏ tới một ZIP BDD100K đã tải sẵn.

In [ ]:
from pathlib import Path

# Đường dẫn lưu trên máy train (Linux).
DATA_ROOT = Path("/mnt/bdd100k")
RUNS_ROOT = DATA_ROOT / "runs"

# Đặt Path cụ thể nếu notebook không được chạy từ bên trong repo ml-fastvit.
REPO_ROOT_OVERRIDE = None  # Ví dụ: Path("/mnt/ml-fastvit")

# Download/convert.
EXISTING_ARCHIVE = None  # Ví dụ: DATA_ROOT / "bdd100k.zip"
CONVERT_ONLY = False     # True nếu ảnh + nhãn gốc đã có trong DATA_ROOT.
KEEP_DOWNLOADED_ZIP = False
INSTALL_DEPENDENCIES = True

# Full run khuyến nghị trong docs/usage_guide.md.
MODEL = "fastvit_sa12"
ARCHITECTURE_VERSION = "fixed_c5"
CLASSIFICATION_LOSS = "qfl"
IMG_SIZE = 800
FPN_CHANNELS = 256
ANCHOR_SIZES = (11, 19, 28, 51, 153)
EPOCHS = 50
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 4
WORKERS = 8
ACCUM_STEPS = 1
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
CLIP_GRAD = 5.0
FOCAL_GAMMA = 2.0
SCHEDULER = "step"
WARMUP_ITERS = 500
WARMUP_EPOCHS = 5
LR_STEPS = (8, 11)
LR_GAMMA = 0.1
USE_AMP = True
CACHE_RAM = False
EVAL_INTERVAL = 1
SCORE_THRESHOLD = 0.05
NMS_THRESHOLD = 0.5
PRE_NMS_TOPK = 2000
MAX_DETECTIONS = 100
SEED = 42
RESUME_CHECKPOINT = None  # Ví dụ: DATA_ROOT / "runs/training/.../last.pth"

# ImageNet-1K pretrained backbone chính thức của Apple (unfused, dùng cho fine-tuning).
USE_PRETRAINED_BACKBONE = True
PRETRAINED_BACKBONE = DATA_ROOT / "pretrained" / "fastvit_sa12.pth.tar"
PRETRAINED_BACKBONE_URL = (
    "https://docs-assets.developer.apple.com/ml-research/models/fastvit/"
    "image_classification_models/fastvit_sa12.pth.tar"
)

# Đánh giá cuối cùng. Đặt số nguyên để thử nhanh trên một phần validation.
RUN_FINAL_EVALUATION = True
MAX_EVAL_SAMPLES = None  # Ví dụ: 500
SAVE_JSON_DURING_TRAIN = True
SAVE_VISUALIZATIONS = True

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"RUNS_ROOT: {RUNS_ROOT}")

In [ ]:
# Tự tìm thư mục gốc của repo bằng dấu hiệu train_bdd100k.py.
import os
import platform
import shutil
import subprocess
import sys

def find_repo_root(override=None):
    if override is not None:
        candidates = [Path(override).expanduser()]
    else:
        cwd = Path.cwd().resolve()
        candidates = [cwd, *cwd.parents, Path("/mnt/ml-fastvit"), Path("/content/ml-fastvit")]
    for candidate in candidates:
        if (candidate / "train_bdd100k.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Không tìm thấy repo ml-fastvit. Hãy đặt REPO_ROOT_OVERRIDE tới thư mục chứa train_bdd100k.py."
    )

REPO_ROOT = find_repo_root(REPO_ROOT_OVERRIDE)
usage = shutil.disk_usage(DATA_ROOT)
free_gib = usage.free / 1024**3

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Python: {sys.version.split()[0]} | OS: {platform.platform()}")
print(f"Dung lượng trống tại {DATA_ROOT}: {free_gib:.1f} GiB")
if free_gib < 30:
    print("CẢNH BÁO: nên có ít nhất khoảng 30 GiB trống trước khi tải và train.")

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True, check=False,
) if shutil.which("nvidia-smi") else None
print("GPU:", gpu_check.stdout.strip() if gpu_check and gpu_check.returncode == 0 else "không tìm thấy nvidia-smi")

## 2. Cài dependencies

Cell này dùng đúng Python của kernel. Sau lần cài đầu có thể đặt `INSTALL_DEPENDENCIES = False` để bỏ qua. Nếu môi trường vừa cài mới PyTorch, có thể cần restart kernel rồi chạy lại từ đầu.

In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Bỏ qua cài dependencies.")

In [ ]:
# Import sau khi cài dependencies và chuẩn bị hàm chạy lệnh có lưu log.
import json
import shlex
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def run_and_tee(command, log_path, cwd=REPO_ROOT):
    command = [str(part) for part in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("$", shlex.join(command), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command, cwd=cwd, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return log_path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU đang dùng: {torch.cuda.get_device_name(0)}")

## 3. Tải và kiểm tra pretrained backbone

Full run dùng checkpoint **FastViT-SA12 ImageNet-1K unfused** từ [Apple FastViT Model Zoo](https://github.com/apple-aiml-research/ml-fastvit#fastvit-model-zoo) để fine-tune cho detection. File được tải một lần vào `/mnt/bdd100k/pretrained/`. Notebook không dùng bản reparameterized/fused vì bản unfused mới tương thích với kiến trúc lúc train.

In [ ]:
import gc
import hashlib

if USE_PRETRAINED_BACKBONE:
    if MODEL != "fastvit_sa12":
        raise ValueError(
            "PRETRAINED_BACKBONE_URL hiện dành cho fastvit_sa12; "
            "hãy đổi URL và tên file nếu đổi MODEL."
        )
    PRETRAINED_BACKBONE = Path(PRETRAINED_BACKBONE).expanduser()
    PRETRAINED_BACKBONE.parent.mkdir(parents=True, exist_ok=True)
    if not PRETRAINED_BACKBONE.is_file():
        partial_path = Path(str(PRETRAINED_BACKBONE) + ".part")
        partial_path.unlink(missing_ok=True)
        print(f"Đang tải pretrained backbone từ {PRETRAINED_BACKBONE_URL}")
        torch.hub.download_url_to_file(
            PRETRAINED_BACKBONE_URL, str(partial_path), progress=True
        )
        partial_path.replace(PRETRAINED_BACKBONE)
    else:
        print(f"Dùng pretrained đã có: {PRETRAINED_BACKBONE}")

    # Preflight: checkpoint phải nạp được và khớp phần lớn tham số backbone.
    import models  # đăng ký các FastViT variant với timm
    from timm.models import create_model

    checkpoint = torch.load(PRETRAINED_BACKBONE, map_location="cpu", weights_only=True)
    # Dùng đúng quy tắc mà FastViTDetector áp dụng khi nhận --pretrained-backbone.
    state_dict = checkpoint.get("state_dict", checkpoint)
    reference_backbone = create_model(
        MODEL, fork_feat=True, feature_version=ARCHITECTURE_VERSION
    )
    reference_state = reference_backbone.state_dict()
    matched = {
        key: value for key, value in state_dict.items()
        if key in reference_state and value.shape == reference_state[key].shape
    }
    matched_numel = sum(reference_state[key].numel() for key in matched)
    total_numel = sum(value.numel() for value in reference_state.values())
    match_ratio = matched_numel / total_numel
    if match_ratio < 0.90:
        raise RuntimeError(
            f"Pretrained chỉ khớp {match_ratio:.1%} tham số; không bắt đầu full run."
        )

    digest = hashlib.sha256()
    with PRETRAINED_BACKBONE.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    pretrained_metadata = {
        "path": str(PRETRAINED_BACKBONE),
        "source": PRETRAINED_BACKBONE_URL,
        "sha256": digest.hexdigest(),
        "matched_keys": len(matched),
        "backbone_keys": len(reference_state),
        "matched_parameter_ratio": match_ratio,
    }
    with (PRETRAINED_BACKBONE.parent / "fastvit_sa12_metadata.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(pretrained_metadata, handle, indent=2)
    print(json.dumps(pretrained_metadata, indent=2))
    del checkpoint, state_dict, reference_backbone, reference_state, matched
    gc.collect()
else:
    PRETRAINED_BACKBONE = None
    raise RuntimeError("Full run yêu cầu USE_PRETRAINED_BACKBONE = True.")

## 4. Tải BDD100K và chuyển sang COCO

Downloader của repo tải mirror trên Kaggle, chỉ giải nén ảnh/nhãn detection cần thiết, chuẩn hoá layout và sinh hai file COCO. Quá trình có marker nên có thể chạy lại an toàn sau khi đã hoàn tất.

Nếu đã có ZIP, đặt `EXISTING_ARCHIVE` ở cell cấu hình. Nếu dữ liệu gốc đã giải nén sẵn, đặt `CONVERT_ONLY = True`.

In [ ]:
download_command = [
    sys.executable, "-m", "tools.download_bdd100k",
    "--dest-dir", DATA_ROOT,
]
if CONVERT_ONLY:
    download_command.append("--convert-only")
elif EXISTING_ARCHIVE is not None:
    download_command.extend(["--archive", Path(EXISTING_ARCHIVE).expanduser()])
elif KEEP_DOWNLOADED_ZIP:
    download_command.append("--keep-zip")

run_and_tee(download_command, DATA_ROOT / "download_and_convert.log")

## 5. Quét các shard ảnh và kiểm tra COCO

Giống `bdd1000k.ipynb`, notebook ưu tiên quét `trainA`, `trainB`, `testA`, `testB` và thư mục train gốc, sau đó quét thêm mọi thư mục con có ảnh. Validation được quét tương tự. Index lưu cả relative path lẫn basename để đối chiếu chính xác với COCO và phát hiện tên ảnh trùng giữa các shard.

In [ ]:
from collections import defaultdict

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
image_roots = {
    split: DATA_ROOT / "images" / "100k" / split
    for split in ("train", "val")
}

def discover_image_directories(root, preferred_names=()):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục ảnh: {root}")
    directories = []
    seen = set()

    def add_if_contains_images(directory):
        directory = Path(directory)
        if directory in seen or not directory.is_dir():
            return
        seen.add(directory)
        if any(
            child.is_file() and child.suffix.lower() in IMAGE_EXTENSIONS
            for child in directory.iterdir()
        ):
            directories.append(directory)

    # Giữ thứ tự shard như notebook bdd1000k.ipynb.
    for name in preferred_names:
        add_if_contains_images(root / name)
    add_if_contains_images(root)
    for directory in sorted(path for path in root.rglob("*") if path.is_dir()):
        add_if_contains_images(directory)
    if not directories:
        raise RuntimeError(f"Không tìm thấy ảnh dưới {root}")
    return directories

TRAIN_DIRS = discover_image_directories(
    image_roots["train"], ("trainA", "trainB", "testA", "testB")
)
VAL_DIRS = discover_image_directories(image_roots["val"], ("valA", "valB"))

def build_image_index(directories, split_root, split_name):
    by_relative = {}
    by_name = defaultdict(list)
    directory_counts = {}
    for directory in directories:
        image_paths = sorted(
            path for path in directory.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        directory_counts[directory] = len(image_paths)
        print(f"Scanning [{split_name}]: {directory} ({len(image_paths):,} ảnh)")
        for image_path in image_paths:
            relative_name = image_path.relative_to(split_root).as_posix()
            by_relative[relative_name] = image_path
            by_name[image_path.name].append(image_path)
    duplicates = {name: paths for name, paths in by_name.items() if len(paths) > 1}
    print(
        f"{split_name}: {len(by_relative):,} ảnh trong {len(directories)} thư mục; "
        f"{len(duplicates):,} basename bị trùng"
    )
    return {
        "by_relative": by_relative,
        "by_name": dict(by_name),
        "duplicates": duplicates,
        "directory_counts": directory_counts,
    }

image_indexes = {
    "train": build_image_index(TRAIN_DIRS, image_roots["train"], "train"),
    "val": build_image_index(VAL_DIRS, image_roots["val"], "val"),
}


In [ ]:
annotation_paths = {
    split: DATA_ROOT / "annotations" / f"bdd100k_det_{split}_coco.json"
    for split in ("train", "val")
}
datasets = {}
resolved_image_paths = {"train": {}, "val": {}}

for split in ("train", "val"):
    ann_path = annotation_paths[split]
    if not ann_path.is_file():
        raise FileNotFoundError(f"Thiếu annotation: {ann_path}")
    with ann_path.open("r", encoding="utf-8") as handle:
        datasets[split] = json.load(handle)
    coco = datasets[split]
    category_ids = {int(item["id"]) for item in coco["categories"]}
    if category_ids != set(range(1, 11)):
        raise ValueError(f"{split}: category IDs phải là 1..10, nhận được {sorted(category_ids)}")
    missing = []
    ambiguous = []
    index = image_indexes[split]
    for item in coco["images"]:
        file_name = item["file_name"].replace("\\", "/")
        image_path = index["by_relative"].get(file_name)
        if image_path is None:
            candidates = index["by_name"].get(Path(file_name).name, [])
            if len(candidates) == 1:
                image_path = candidates[0]
            elif len(candidates) > 1:
                ambiguous.append((file_name, [str(path) for path in candidates]))
            else:
                missing.append(file_name)
        if image_path is not None:
            resolved_image_paths[split][item["file_name"]] = image_path
    if missing:
        raise FileNotFoundError(f"{split}: thiếu {len(missing)} ảnh, ví dụ {missing[:3]}")
    if ambiguous:
        raise ValueError(f"{split}: {len(ambiguous)} tên ảnh mơ hồ giữa các shard, ví dụ {ambiguous[:1]}")
    print(
        f"{split:>5}: {len(coco['images']):,} ảnh | "
        f"{len(coco['annotations']):,} bbox | 10 categories | không thiếu ảnh"
    )

print("Categories:", {item["id"]: item["name"] for item in datasets["train"]["categories"]})

In [ ]:
# Hiển thị ảnh train đầu tiên có annotation.
train_coco = datasets["train"]
image_by_id = {item["id"]: item for item in train_coco["images"]}
category_by_id = {item["id"]: item["name"] for item in train_coco["categories"]}
annotations_by_image = {}
for annotation in train_coco["annotations"]:
    annotations_by_image.setdefault(annotation["image_id"], []).append(annotation)
sample_id = next(image_id for image_id in image_by_id if annotations_by_image.get(image_id))
sample_info = image_by_id[sample_id]
sample_path = resolved_image_paths["train"][sample_info["file_name"]]
sample = Image.open(sample_path).convert("RGB")
canvas = sample.copy()
draw = ImageDraw.Draw(canvas)
for annotation in annotations_by_image[sample_id]:
    x, y, width, height = annotation["bbox"]
    draw.rectangle((x, y, x + width, y + height), outline="red", width=3)
    draw.text((x + 3, y + 3), category_by_id[annotation["category_id"]], fill="yellow")
plt.figure(figsize=(16, 9))
plt.imshow(canvas)
plt.title(f"{sample_info['file_name']} — {len(annotations_by_image[sample_id])} boxes")
plt.axis("off")
plt.show()

## 6. Xác nhận anchor của full run

Full run dùng đúng năm anchor P2–P6 `11 19 28 51 153` trong `docs/usage_guide.md`. Cấu hình được lưu tại `/mnt/bdd100k/anchor_config.json` và dùng nhất quán cho train, resume và evaluation.

In [ ]:
if len(ANCHOR_SIZES) != 5 or any(size <= 0 for size in ANCHOR_SIZES):
    raise ValueError(f"ANCHOR_SIZES không hợp lệ: {ANCHOR_SIZES}")
anchor_config = {
    "source": "docs/usage_guide.md - cấu hình full run khuyến nghị",
    "img_size": IMG_SIZE,
    "anchor_sizes": ANCHOR_SIZES,
    "fpn_levels": ["P2", "P3", "P4", "P5", "P6"],
}
with (DATA_ROOT / "anchor_config.json").open("w", encoding="utf-8") as handle:
    json.dump(anchor_config, handle, indent=2)
print(json.dumps(anchor_config, indent=2))

## 7. Train FastViT — full config + pretrained

Mỗi run tạo một thư mục timestamp riêng dưới `/mnt/bdd100k/runs/training`. Checkpoint `last.pth` được ghi mỗi epoch; `best.pth` được cập nhật khi validation mAP tốt hơn. Để tiếp tục run bị ngắt, đặt `RESUME_CHECKPOINT` ở cell cấu hình rồi chạy lại cell train.

In [ ]:
training_root = RUNS_ROOT / "training"
training_root.mkdir(parents=True, exist_ok=True)
if PRETRAINED_BACKBONE is None or not Path(PRETRAINED_BACKBONE).is_file():
    raise FileNotFoundError("Full run yêu cầu pretrained backbone hợp lệ.")
if RESUME_CHECKPOINT is not None and not Path(RESUME_CHECKPOINT).expanduser().is_file():
    raise FileNotFoundError(f"Không tìm thấy checkpoint resume: {RESUME_CHECKPOINT}")
train_command = [
    sys.executable, REPO_ROOT / "train_bdd100k.py",
    "--data-dir", DATA_ROOT,
    "--model", MODEL,
    "--architecture-version", ARCHITECTURE_VERSION,
    "--classification-loss", CLASSIFICATION_LOSS,
    "--img-size", IMG_SIZE,
    "--fpn-channels", FPN_CHANNELS,
    "--anchor-sizes", *ANCHOR_SIZES,
    "--epochs", EPOCHS,
    "--batch-size", BATCH_SIZE,
    "--eval-batch-size", EVAL_BATCH_SIZE,
    "--workers", WORKERS,
    "--accum-steps", ACCUM_STEPS,
    "--lr", LEARNING_RATE,
    "--weight-decay", WEIGHT_DECAY,
    "--clip-grad", CLIP_GRAD,
    "--focal-gamma", FOCAL_GAMMA,
    "--scheduler", SCHEDULER,
    "--warmup-iters", WARMUP_ITERS,
    "--warmup-epochs", WARMUP_EPOCHS,
    "--lr-steps", *LR_STEPS,
    "--lr-gamma", LR_GAMMA,
    "--eval-interval", EVAL_INTERVAL,
    "--score-thresh", SCORE_THRESHOLD,
    "--nms-thresh", NMS_THRESHOLD,
    "--pre-nms-topk", PRE_NMS_TOPK,
    "--max-detections", MAX_DETECTIONS,
    "--pretrained-backbone", PRETRAINED_BACKBONE,
    "--seed", SEED,
    "--output", training_root,
]
train_command.append("--amp" if USE_AMP else "--no-amp")
if SAVE_JSON_DURING_TRAIN:
    train_command.append("--save-json")
if CACHE_RAM:
    train_command.append("--cache-ram")
if RESUME_CHECKPOINT is not None:
    train_command.extend(["--resume", Path(RESUME_CHECKPOINT).expanduser()])

train_log = DATA_ROOT / f"train_{MODEL}_{time.strftime('%Y%m%d_%H%M%S')}.log"
run_and_tee(train_command, train_log)

run_directories = [path for path in training_root.glob(f"{MODEL}_*") if path.is_dir()]
if not run_directories:
    raise RuntimeError(f"Không tìm thấy thư mục run trong {training_root}")
TRAIN_RUN_DIR = max(run_directories, key=lambda path: path.stat().st_mtime)
BEST_CHECKPOINT = TRAIN_RUN_DIR / "best.pth"
if not BEST_CHECKPOINT.is_file():
    BEST_CHECKPOINT = TRAIN_RUN_DIR / "last.pth"
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Không tìm thấy best.pth hoặc last.pth trong {TRAIN_RUN_DIR}")
print(f"Run: {TRAIN_RUN_DIR}")
print(f"Checkpoint dùng để đánh giá: {BEST_CHECKPOINT}")
print(f"Log: {train_log}")

## 8. Đánh giá checkpoint tốt nhất

Cell này chạy COCO evaluation độc lập trên validation, lưu JSON dự đoán/ground truth và (nếu bật) ảnh trực quan hoá dưới `/mnt/bdd100k/runs/evaluation`.

In [ ]:
if RUN_FINAL_EVALUATION:
    evaluation_root = RUNS_ROOT / "evaluation"
    json_output_root = evaluation_root / "json_results"
    evaluation_root.mkdir(parents=True, exist_ok=True)
    json_output_root.mkdir(parents=True, exist_ok=True)
    eval_command = [
        sys.executable, REPO_ROOT / "train_bdd100k.py",
        "--data-dir", DATA_ROOT,
        "--model", MODEL,
        "--architecture-version", ARCHITECTURE_VERSION,
        "--classification-loss", CLASSIFICATION_LOSS,
        "--img-size", IMG_SIZE,
        "--fpn-channels", FPN_CHANNELS,
        "--anchor-sizes", *ANCHOR_SIZES,
        "--batch-size", BATCH_SIZE,
        "--eval-batch-size", EVAL_BATCH_SIZE,
        "--workers", WORKERS,
        "--focal-gamma", FOCAL_GAMMA,
        "--score-thresh", SCORE_THRESHOLD,
        "--nms-thresh", NMS_THRESHOLD,
        "--pre-nms-topk", PRE_NMS_TOPK,
        "--max-detections", MAX_DETECTIONS,
        "--resume", BEST_CHECKPOINT,
        "--eval-only",
        "--save-json",
        "--json-output-dir", json_output_root,
        "--output", evaluation_root,
    ]
    eval_command.append("--amp" if USE_AMP else "--no-amp")
    if MAX_EVAL_SAMPLES is not None:
        eval_command.extend(["--max-eval-samples", MAX_EVAL_SAMPLES])
    if SAVE_VISUALIZATIONS:
        eval_command.append("--save-visualizations")
    eval_log = DATA_ROOT / f"evaluate_{MODEL}_{time.strftime('%Y%m%d_%H%M%S')}.log"
    run_and_tee(eval_command, eval_log)
    print(f"Log đánh giá: {eval_log}")
    print(f"JSON kết quả: {json_output_root}")
else:
    print("Bỏ qua đánh giá cuối cùng.")

## 9. Tổng kết artifact

Có thể dùng `best.pth` để inference/evaluate sau này. `args.json` và `protocol.json` trong thư mục run ghi lại toàn bộ cấu hình, metadata kiến trúc, môi trường và phiên bản mã nguồn. Metadata nguồn, SHA-256 và tỷ lệ weights pretrained khớp được lưu cạnh file pretrained.

In [ ]:
artifacts = {
    "data_root": str(DATA_ROOT),
    "train_annotations": str(annotation_paths["train"]),
    "val_annotations": str(annotation_paths["val"]),
    "pretrained_backbone": str(PRETRAINED_BACKBONE),
    "pretrained_metadata": str(Path(PRETRAINED_BACKBONE).parent / "fastvit_sa12_metadata.json"),
    "anchor_config": str(DATA_ROOT / "anchor_config.json"),
    "train_run": str(TRAIN_RUN_DIR),
    "checkpoint": str(BEST_CHECKPOINT),
}
print(json.dumps(artifacts, indent=2, ensure_ascii=False))